In [ ]:
import os
import cv2
import yaml
import time
import random
import numpy as np
import pandas as pd
import pydicom
import albumentations as A

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed


# Config

In [ ]:
# =========================
# PATH CONFIG
# =========================
from pathlib import Path

ROOT = Path("/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection")

TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"

# Label CSV giống LUNG_YOLO_CODE/train.py: train+test đã merge sẵn, có khoảng 1000 No finding.
LABEL_CSV_CANDIDATES = [
    Path('/kaggle/input/datasets/benxelua/final-label/annotations_all_merged_other_1000nf.csv'),
    Path('/kaggle/input/clean-dataset/annotations_all_merged_other_1000nf.csv'),
    Path('/kaggle/input/lung-yolo-code/annotations_all_merged_other_1000nf.csv'),
    Path('/kaggle/input/LUNG_YOLO_CODE/annotations_all_merged_other_1000nf.csv'),
    Path('LUNG_YOLO_CODE/annotations_all_merged_other_1000nf.csv'),
]
LABEL_CSV = next((p for p in LABEL_CSV_CANDIDATES if p.exists()), LABEL_CSV_CANDIDATES[0])

# output YOLO datasets, one subfolder per 3-channel preprocessing combo
OUT_ROOT = Path("/kaggle/working/yolo_vindr_multiclass_combos")

# =========================
# SUBSET CONFIG
# =========================
USE_SMALL_SUBSET = False     # True = subset nhỏ, False = full dataset
IMAGES_PER_CLASS = 2         # mỗi class lấy khoảng N ảnh trong small subset
NO_FINDING_IMAGES = None     # None = giữ đúng toàn bộ No finding có trong LABEL_CSV 1000nf
SEED = 42

# Giống CSV runtime flow trong LUNG_YOLO_CODE/train.py
SPLIT_RATIOS = {
    "train": 0.70,
    "val": 0.10,
    "test": 0.20,
}

# =========================
# IMAGE / PREPROCESS CONFIG
# =========================
IMG_SIZE = 512
SAVE_AS_JPG = False          # yêu cầu hiện tại: export PNG 3-channel
JPG_QUALITY = 95
REBUILD_OUTPUT = True        # True = xóa dataset combo cũ trong OUT_ROOT trước khi export lại
CLEANUP_AFTER_COMBO = True      # True = train/eval xong combo nào thì xóa images/labels combo đó để tiết kiệm disk
ENABLE_TRAIN_AUG = True

LOW_PERCENTILE = 0.5
HIGH_PERCENTILE = 99.5

PREPROCESS_COMBOS = [
    ("raw_minmax", "voi_lut", "percentile"),
    ("raw_minmax", "voi_lut", "clahe"),
    ("raw_minmax", "percentile", "clahe"),
    ("voi_lut", "percentile", "clahe"),
]

# =========================
# TRAIN CONFIG
# =========================
MODEL_NAME = "yolov8n.pt"
TRAIN_EPOCHS = 50
TRAIN_BATCH = 8
EVAL_BATCH = 8

# Helper functions

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)


def find_dicom_files_for_image_ids(folder: Path, image_ids):
    requested_ids = sorted({str(x).strip() for x in image_ids if pd.notna(x) and str(x).strip()})
    matched = {}

    for image_id in requested_ids:
        direct_candidates = [
            folder / f"{image_id}.dicom",
            folder / f"{image_id}.dcm",
            folder / image_id,
        ]
        for candidate in direct_candidates:
            if candidate.is_file():
                matched[image_id] = candidate
                break

    remaining = [image_id for image_id in requested_ids if image_id not in matched]
    if remaining:
        remaining_set = set(remaining)
        for p in sorted(folder.rglob("*")):
            if not remaining_set:
                break
            if p.is_file() and p.suffix.lower() in {".dicom", ".dcm"} and p.stem in remaining_set:
                matched[p.stem] = p
                remaining_set.remove(p.stem)

    missing = [image_id for image_id in requested_ids if image_id not in matched]
    matched_files = [matched[image_id] for image_id in requested_ids if image_id in matched]
    return matched_files, matched, missing


from pathlib import Path
import numpy as np
import pydicom
from pydicom.multival import MultiValue
from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut


def normalize_to_uint8(img):
    img = np.asarray(img, dtype=np.float32)
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    img_min = float(np.min(img))
    img_max = float(np.max(img))
    if img_max > img_min:
        img = (img - img_min) / (img_max - img_min)
    else:
        img = np.zeros_like(img, dtype=np.float32)
    return (img * 255.0).clip(0, 255).astype(np.uint8)


def fix_monochrome(img, ds):
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = np.max(img) - img
    return img


def dicom_rescale_pixels(ds):
    img = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    return img * slope + intercept


def first_float(value, default):
    if value is None:
        return float(default)
    if hasattr(value, "value"):
        value = value.value
    if isinstance(value, MultiValue) or isinstance(value, (list, tuple, np.ndarray)):
        if len(value) == 0:
            return float(default)
        value = value[0]
    try:
        return float(value)
    except Exception:
        return float(default)


def raw_minmax_from_ds(ds):
    img = ds.pixel_array.astype(np.float32)
    img = fix_monochrome(img, ds)
    return normalize_to_uint8(img)


def voi_lut_from_ds(ds):
    img = apply_modality_lut(ds.pixel_array, ds)
    img = apply_voi_lut(img, ds).astype(np.float32)
    img = fix_monochrome(img, ds)
    return normalize_to_uint8(img)


def percentile_from_ds(ds, low=LOW_PERCENTILE, high=HIGH_PERCENTILE):
    img = dicom_rescale_pixels(ds)
    img = fix_monochrome(img, ds)

    lo = float(np.percentile(img, low))
    hi = float(np.percentile(img, high))
    if hi <= lo:
        hi = lo + 1.0

    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo)
    return (img * 255.0).clip(0, 255).astype(np.uint8)


def clahe_from_ds(ds):
    img = dicom_rescale_pixels(ds)
    img = fix_monochrome(img, ds)

    default_center = float(np.median(img))
    default_width = float(np.std(img) * 4.0)
    c_orig = first_float(ds.get("WindowCenter", default_center), default_center)
    w_orig = first_float(ds.get("WindowWidth", default_width), default_width)

    if w_orig < 1:
        w_orig = float(np.std(img) * 4.0 + 1e-6)

    img_min = c_orig - w_orig / 2.0
    img_max = c_orig + w_orig / 2.0
    if img_max <= img_min:
        img_min = float(np.min(img))
        img_max = float(np.max(img))
        if img_max <= img_min:
            return np.zeros_like(img, dtype=np.uint8)

    img_windowed = np.clip(img, img_min, img_max)
    img_8bit = ((img_windowed - img_min) / (img_max - img_min) * 255.0).clip(0, 255).astype(np.uint8)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img_8bit)


# Các hàm path-level để có thể test riêng từng cách xử lý.
def read_dicom_raw_minmax(path: Path):
    ds = pydicom.dcmread(str(path))
    return raw_minmax_from_ds(ds)


def read_dicom_to_uint8(path: Path):
    # Backward-compatible alias: raw min-max theo pixel gốc.
    return read_dicom_raw_minmax(path)


def read_dicom_voi_lut(path: Path):
    ds = pydicom.dcmread(str(path))
    return voi_lut_from_ds(ds)


def read_dicom_percentile_clipping(path: Path, low=LOW_PERCENTILE, high=HIGH_PERCENTILE):
    ds = pydicom.dcmread(str(path))
    return percentile_from_ds(ds, low=low, high=high)


def read_dicom_with_clahe(path: Path):
    ds = pydicom.dcmread(str(path))
    img_clahe = clahe_from_ds(ds)
    return cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2BGR)


def read_dicom_preprocess_variants(path: Path):
    ds = pydicom.dcmread(str(path))
    return {
        "raw_minmax": raw_minmax_from_ds(ds),
        "voi_lut": voi_lut_from_ds(ds),
        "percentile": percentile_from_ds(ds, low=LOW_PERCENTILE, high=HIGH_PERCENTILE),
        "clahe": clahe_from_ds(ds),
    }


def combo_to_name(combo):
    return "__".join(combo)


def build_preprocess_combo_image(variants, combo):
    channels = []
    for method_name in combo:
        channel = variants[method_name]
        if channel.ndim == 3:
            channel = cv2.cvtColor(channel, cv2.COLOR_BGR2GRAY)
        channels.append(channel.astype(np.uint8))
    return np.stack(channels, axis=-1)


def resize_image_keep_shape(img, size=1024):
    h, w = img.shape[:2]
    resized = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return resized, w, h

# Đọc CSV và chuẩn hóa annotation

In [ ]:
# Label fusion helpers for VinDr-CXR training annotations
from dataclasses import dataclass

@dataclass(frozen=True)
class FusionConfig:
    iou_threshold: float = 0.5
    ios_threshold: float = 0.7
    merge_mode: str = "median"  # options: median, mean, enclosing


def _box_area(box):
    width = max(0.0, float(box[2]) - float(box[0]))
    height = max(0.0, float(box[3]) - float(box[1]))
    return width * height


def _intersection_area(box_a, box_b):
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    width = max(0.0, x2 - x1)
    height = max(0.0, y2 - y1)
    return width * height


def compute_iou(box_a, box_b):
    inter = _intersection_area(box_a, box_b)
    if inter <= 0.0:
        return 0.0
    union = _box_area(box_a) + _box_area(box_b) - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def compute_ios(box_a, box_b):
    inter = _intersection_area(box_a, box_b)
    if inter <= 0.0:
        return 0.0
    smaller = min(_box_area(box_a), _box_area(box_b))
    if smaller <= 0.0:
        return 0.0
    return inter / smaller


def should_link_boxes(box_a, box_b, config):
    return (
        compute_iou(box_a, box_b) >= config.iou_threshold
        or compute_ios(box_a, box_b) >= config.ios_threshold
    )


def merge_cluster_boxes(boxes, merge_mode="median"):
    boxes = np.asarray(boxes, dtype=float)
    if len(boxes) == 1:
        return boxes[0].astype(float)

    if merge_mode == "median":
        merged = np.median(boxes, axis=0)
    elif merge_mode == "mean":
        merged = np.mean(boxes, axis=0)
    elif merge_mode == "enclosing":
        merged = np.array([
            np.min(boxes[:, 0]),
            np.min(boxes[:, 1]),
            np.max(boxes[:, 2]),
            np.max(boxes[:, 3]),
        ], dtype=float)
    else:
        raise ValueError(f"Unsupported merge_mode: {merge_mode}")

    if merged[2] <= merged[0]:
        merged[2] = merged[0] + 1e-6
    if merged[3] <= merged[1]:
        merged[3] = merged[1] + 1e-6
    return merged.astype(float)


def _connected_components(indices, adjacency):
    components = []
    visited = set()

    for start in indices:
        if start in visited:
            continue
        stack = [start]
        component = []
        visited.add(start)

        while stack:
            node = stack.pop()
            component.append(node)
            for neighbor in adjacency[node]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)

        components.append(sorted(component))

    return components


def fuse_same_class_boxes(class_df, bbox_cols=("x_min", "y_min", "x_max", "y_max"), config=None):
    config = config or FusionConfig()
    if len(class_df) <= 1:
        out = class_df.copy()
        out["fusion_group_size"] = 1
        out["was_fused"] = False
        return out

    coords = class_df.loc[:, list(bbox_cols)].to_numpy(dtype=float)
    indices = list(range(len(class_df)))
    adjacency = {idx: set() for idx in indices}

    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            if should_link_boxes(coords[i], coords[j], config):
                adjacency[i].add(j)
                adjacency[j].add(i)

    components = _connected_components(indices, adjacency)
    fused_rows = []

    for component in components:
        component_df = class_df.iloc[component].copy()
        merged_box = merge_cluster_boxes(
            component_df.loc[:, list(bbox_cols)].to_numpy(dtype=float),
            merge_mode=config.merge_mode,
        )

        row = component_df.iloc[0].copy()
        row[bbox_cols[0]] = float(merged_box[0])
        row[bbox_cols[1]] = float(merged_box[1])
        row[bbox_cols[2]] = float(merged_box[2])
        row[bbox_cols[3]] = float(merged_box[3])
        row["fusion_group_size"] = int(len(component))
        row["was_fused"] = bool(len(component) > 1)
        fused_rows.append(row)

    return pd.DataFrame(fused_rows).reset_index(drop=True)


def fuse_annotation_dataframe(
    df,
    image_col="image_id",
    class_col="class_name",
    bbox_cols=("x_min", "y_min", "x_max", "y_max"),
    config=None,
):
    config = config or FusionConfig()
    fused_groups = []
    grouped = df.groupby([image_col, class_col], sort=False, group_keys=False)
    for _, group_df in grouped:
        fused_groups.append(
            fuse_same_class_boxes(
                class_df=group_df,
                bbox_cols=bbox_cols,
                config=config,
            )
        )

    if not fused_groups:
        out = df.copy()
        out["fusion_group_size"] = 1
        out["was_fused"] = False
        return out

    return pd.concat(fused_groups, ignore_index=True).reset_index(drop=True)



In [ ]:
# Label handling ported from LUNG_YOLO_CODE/train.py CSV runtime flow.
CSV_BACKGROUND_CLASS = "No finding"
CSV_BBOX_FORMAT = "auto"   # auto, xyxy_pixel, xyxy_norm
CSV_SCORE_COL = "score"
CSV_IMAGE_WIDTH_COL = ""
CSV_IMAGE_HEIGHT_COL = ""
WBF_IOU_THRESHOLD = 0.5
WBF_SKIP_BOX_THRESHOLD = 0.0


def is_background_class(class_name):
    normalized = str(class_name).strip().lower().replace("_", " ")
    background_names = {
        CSV_BACKGROUND_CLASS.strip().lower().replace("_", " "),
        "no finding",
        "background",
        "negative",
    }
    return normalized in background_names


def first_existing_column(df, explicit_col, candidates):
    if explicit_col and explicit_col in df.columns:
        return explicit_col
    for col in candidates:
        if col in df.columns:
            return col
    return None


def normalize_label_csv(csv_path: Path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        candidate_text = "\n".join(f"- {p}" for p in LABEL_CSV_CANDIDATES)
        raise FileNotFoundError(
            f"Không thấy LABEL_CSV merged 1000nf: {csv_path}\n"
            f"Các path đã thử:\n{candidate_text}"
        )

    out = pd.read_csv(csv_path)
    print("CSV label path:", csv_path)
    print("CSV shape:", out.shape)
    print("CSV columns:", out.columns.tolist())
    display(out.head())

    required_cols = ["image_id", "class_name", "x_min", "y_min", "x_max", "y_max"]
    missing_cols = [c for c in required_cols if c not in out.columns]
    if missing_cols:
        raise ValueError(f"CSV thiếu cột bắt buộc: {missing_cols}")

    out = out.copy()
    out["image_id"] = out["image_id"].astype("string").str.strip()
    out["class_name"] = out["class_name"].astype("string").str.strip()
    out = out.dropna(subset=["image_id", "class_name"]).copy()

    source_col = first_existing_column(out, "source", ["source", "original_split", "split"])
    if source_col:
        out["source"] = out[source_col].astype("string").str.strip().str.lower()
        print("CSV source column:", source_col)
    else:
        out["source"] = "train"
        print("CSV source column: not found; fallback source=train")

    # DICOM competition chỉ có ROOT/train và ROOT/test. Nếu CSV có val/validation thì đọc từ ROOT/test.
    out["source"] = out["source"].replace({"validation": "test", "valid": "test", "val": "test"})

    sample_id_col = first_existing_column(out, "sample_id", ["sample_id"])
    if sample_id_col:
        base_sample_id = out[sample_id_col].astype("string").str.strip().astype(str)
    else:
        base_sample_id = out["image_id"].astype(str)

    # Notebook dùng source__id để tránh trùng tên khi đọc DICOM từ 2 folder gốc.
    if base_sample_id.str.contains("__", regex=False).any():
        out["sample_id"] = base_sample_id
    else:
        out["sample_id"] = out["source"].astype(str) + "__" + base_sample_id

    filename_col = first_existing_column(out, "", ["file_name", "filename", "image_path", "path"])
    out["csv_file_name"] = out[filename_col].astype("string").str.strip() if filename_col else ""

    width_col = first_existing_column(out, CSV_IMAGE_WIDTH_COL, ["image_width", "img_width", "original_width", "orig_width", "width"])
    height_col = first_existing_column(out, CSV_IMAGE_HEIGHT_COL, ["image_height", "img_height", "original_height", "orig_height", "height"])
    if width_col and height_col:
        out["source_width"] = pd.to_numeric(out[width_col], errors="coerce")
        out["source_height"] = pd.to_numeric(out[height_col], errors="coerce")
        print("CSV image size columns:", width_col, height_col)
    else:
        out["source_width"] = np.nan
        out["source_height"] = np.nan
        print("CSV image size columns: not found; pixel boxes will use DICOM size")

    if CSV_SCORE_COL and CSV_SCORE_COL in out.columns:
        out["box_score"] = pd.to_numeric(out[CSV_SCORE_COL], errors="coerce").fillna(1.0)
        print("CSV WBF score column:", CSV_SCORE_COL)
    else:
        out["box_score"] = 1.0
        print("CSV WBF score column: not found; all boxes weight=1")

    for c in ["x_min", "y_min", "x_max", "y_max"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out["is_background"] = out["class_name"].map(is_background_class)
    return out.reset_index(drop=True)


def compute_iou_xyxy(box_a, box_b):
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    if inter <= 0.0:
        return 0.0
    area_a = max(0.0, float(box_a[2]) - float(box_a[0])) * max(0.0, float(box_a[3]) - float(box_a[1]))
    area_b = max(0.0, float(box_b[2]) - float(box_b[0])) * max(0.0, float(box_b[3]) - float(box_b[1]))
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def weighted_fuse_group(group_df):
    if len(group_df) <= 1:
        out = group_df.copy()
        out["fusion_group_size"] = 1
        out["was_fused"] = False
        return out

    rows = []
    work = group_df.copy()
    work["_score_sort"] = pd.to_numeric(work["box_score"], errors="coerce").fillna(1.0)
    work = work.sort_values("_score_sort", ascending=False).reset_index(drop=True)

    clusters = []
    for idx, row in work.iterrows():
        box = row[["x_min", "y_min", "x_max", "y_max"]].to_numpy(dtype=float)
        score = float(row.get("box_score", 1.0))
        best_cluster = None
        best_iou = 0.0

        for cluster_idx, cluster in enumerate(clusters):
            iou = compute_iou_xyxy(box, cluster["box"])
            if iou >= WBF_IOU_THRESHOLD and iou > best_iou:
                best_iou = iou
                best_cluster = cluster_idx

        if best_cluster is None:
            clusters.append({
                "indices": [idx],
                "boxes": [box],
                "scores": [score],
                "box": box.copy(),
            })
            continue

        cluster = clusters[best_cluster]
        cluster["indices"].append(idx)
        cluster["boxes"].append(box)
        cluster["scores"].append(score)
        weights = np.asarray(cluster["scores"], dtype=float)
        weights = np.maximum(weights, 1e-6)
        cluster["box"] = np.average(np.asarray(cluster["boxes"], dtype=float), axis=0, weights=weights)

    for cluster in clusters:
        fused = work.loc[cluster["indices"]].iloc[0].copy()
        weights = np.asarray(cluster["scores"], dtype=float)
        weights = np.maximum(weights, 1e-6)
        fused_box = np.average(np.asarray(cluster["boxes"], dtype=float), axis=0, weights=weights)
        fused["x_min"] = float(fused_box[0])
        fused["y_min"] = float(fused_box[1])
        fused["x_max"] = float(fused_box[2])
        fused["y_max"] = float(fused_box[3])
        fused["box_score"] = float(np.mean(cluster["scores"]))
        fused["fusion_group_size"] = int(len(cluster["indices"]))
        fused["was_fused"] = bool(len(cluster["indices"]) > 1)
        rows.append(fused.drop(labels=["_score_sort"], errors="ignore"))

    return pd.DataFrame(rows)


def apply_wbf_to_annotations(ann_df):
    fused_groups = []
    grouped = ann_df.groupby(["sample_id", "class_name"], sort=False, group_keys=False)
    for _, group in grouped:
        fused_groups.append(weighted_fuse_group(group))

    if not fused_groups:
        return ann_df.copy()

    fused_df = pd.concat(fused_groups, ignore_index=True).reset_index(drop=True)
    print("WBF enabled")
    print("WBF IoU threshold:", WBF_IOU_THRESHOLD)
    print("Boxes before WBF:", len(ann_df))
    print("Boxes after WBF:", len(fused_df))
    print("Rows created from fused clusters:", int(fused_df.get("was_fused", pd.Series(dtype=bool)).sum()))
    return fused_df


df_csv_all = normalize_label_csv(LABEL_CSV)

all_sample_ids = set(df_csv_all["sample_id"].dropna().astype(str).str.strip().unique().tolist())
all_csv_ids = all_sample_ids
no_finding_sample_ids = set(
    df_csv_all.loc[df_csv_all["is_background"], "sample_id"].dropna().astype(str).tolist()
)

print("Merged 1000nf CSV shape:", df_csv_all.shape)
print("Merged 1000nf unique sample_id:", len(all_sample_ids))
print("No finding/background sample_id in LABEL_CSV:", len(no_finding_sample_ids))
print("Source distribution from LABEL_CSV:")
display(df_csv_all.drop_duplicates("sample_id")["source"].value_counts().rename_axis("source").reset_index(name="images"))

ann_df = df_csv_all[~df_csv_all["is_background"]].copy()
ann_df = ann_df.dropna(subset=["image_id", "sample_id", "class_name", "x_min", "y_min", "x_max", "y_max"]).copy()
ann_df = ann_df[
    (ann_df["x_max"] > ann_df["x_min"])
    & (ann_df["y_max"] > ann_df["y_min"])
    & (ann_df["box_score"] >= WBF_SKIP_BOX_THRESHOLD)
].copy()

df_before_fusion = ann_df.copy()
df_full = apply_wbf_to_annotations(ann_df)

CLASS_NAMES = sorted(df_full["class_name"].astype(str).unique().tolist())
if not CLASS_NAMES:
    raise ValueError("Không có class bất thường hợp lệ sau khi bỏ No finding/background")
CLASS2ID = {name: i for i, name in enumerate(CLASS_NAMES)}
ID2CLASS = {i: name for name, i in CLASS2ID.items()}

abnormal_sample_ids = set(df_full["sample_id"].astype(str).unique())
background_sample_ids = sorted(all_sample_ids - abnormal_sample_ids)
no_finding_only_sample_ids = sorted(set(no_finding_sample_ids) & set(background_sample_ids))

print("Num classes:", len(CLASS_NAMES))
print("CLASS_NAMES:", CLASS_NAMES)
print("CLASS2ID:", CLASS2ID)
print("Total unique sample_id in LABEL_CSV:", len(all_sample_ids))
print("Images with >=1 valid abnormal bbox:", len(abnormal_sample_ids))
print("Background-only sample_id:", len(background_sample_ids))
print("No finding only sample_id:", len(no_finding_only_sample_ids))
display(
    df_full.loc[df_full.get("was_fused", False), ["source", "image_id", "sample_id", "class_name", "fusion_group_size"]]
    .head(10)
    .reset_index(drop=True)
)

In [ ]:
# # Debug visualize label fusion before vs after for one training image
# import matplotlib.pyplot as plt
# from matplotlib.patches import Rectangle

# DEBUG_FUSION_IMAGE_ID = ""  # ví dụ: "0005e8e3701dfb1dd93d53e2ff537b6e"
# SHOW_CLASS_TEXT = True
# FIGSIZE = (18, 9)

# if not DEBUG_FUSION_IMAGE_ID:
#     raise ValueError("Điền DEBUG_FUSION_IMAGE_ID trước khi chạy cell này")
# if 'df_before_fusion' not in globals():
#     raise ValueError("Chạy cell fusion trước để tạo df_before_fusion và df_full")


# def find_train_dicom_path(train_dir: Path, image_id: str):
#     direct_candidates = [
#         train_dir / f"{image_id}.dicom",
#         train_dir / f"{image_id}.dcm",
#         train_dir / image_id,
#     ]
#     for candidate in direct_candidates:
#         if candidate.is_file():
#             return candidate

#     for p in sorted(train_dir.rglob("*")):
#         if p.is_file() and p.suffix.lower() in {".dicom", ".dcm"} and p.stem == image_id:
#             return p
#     return None


# def draw_boxes(ax, boxes_df, color, title_prefix=""):
#     for _, row in boxes_df.iterrows():
#         x1, y1, x2, y2 = float(row['x_min']), float(row['y_min']), float(row['x_max']), float(row['y_max'])
#         rect = Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2)
#         ax.add_patch(rect)
#         if SHOW_CLASS_TEXT:
#             label = str(row['class_name'])
#             if 'fusion_group_size' in row and pd.notna(row.get('fusion_group_size', np.nan)):
#                 if int(row.get('fusion_group_size', 1)) > 1:
#                     label = f"{label} (n={int(row['fusion_group_size'])})"
#             ax.text(
#                 x1,
#                 max(5.0, y1 - 4.0),
#                 f"{title_prefix}{label}",
#                 color=color,
#                 fontsize=9,
#                 bbox=dict(facecolor='black', alpha=0.5, pad=1),
#             )


# before_df = df_before_fusion[df_before_fusion['image_id'].astype(str) == str(DEBUG_FUSION_IMAGE_ID)].copy()
# after_df = df_full[df_full['image_id'].astype(str) == str(DEBUG_FUSION_IMAGE_ID)].copy()

# print(f"image_id = {DEBUG_FUSION_IMAGE_ID}")
# print(f"Boxes before fusion: {len(before_df)}")
# print(f"Boxes after fusion: {len(after_df)}")

# if len(before_df) == 0 and len(after_df) == 0:
#     raise ValueError(f"Không tìm thấy annotation cho image_id={DEBUG_FUSION_IMAGE_ID}")

# display(before_df[['image_id', 'class_name', 'x_min', 'y_min', 'x_max', 'y_max']].reset_index(drop=True))
# display(after_df[['image_id', 'class_name', 'x_min', 'y_min', 'x_max', 'y_max', 'fusion_group_size', 'was_fused']].reset_index(drop=True))

# dicom_path = find_train_dicom_path(TRAIN_DIR, str(DEBUG_FUSION_IMAGE_ID))
# if dicom_path is None:
#     raise FileNotFoundError(f"Không tìm thấy file DICOM cho image_id={DEBUG_FUSION_IMAGE_ID} trong {TRAIN_DIR}")

# img = read_dicom_to_uint8(dicom_path)

# fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
# for ax in axes:
#     if img.ndim == 2:
#         ax.imshow(img, cmap='gray')
#     else:
#         ax.imshow(img)
#     ax.axis('off')

# axes[0].set_title(f"Before fusion: {DEBUG_FUSION_IMAGE_ID}")
# draw_boxes(axes[0], before_df, color='red', title_prefix='before: ')

# axes[1].set_title(f"After fusion: {DEBUG_FUSION_IMAGE_ID}")
# draw_boxes(axes[1], after_df, color='lime', title_prefix='after: ')

# plt.tight_layout()
# plt.show()



# Chọn subset nhỏ theo từng class từ train + test đã gộp

In [ ]:
import pandas as pd
import numpy as np

print("Building subset from merged train + test CSV...")

# subset theo annotation bbox hợp lệ, nhưng full mode vẫn lấy toàn bộ sample_id từ CSV.
df_work = df_full.copy()

class_to_ids = (
    df_work.groupby("class_name")["sample_id"]
    .unique()
    .to_dict()
)


def build_small_subset_by_class_fast(class_to_ids, images_per_class=12, seed=42, background_ids=None):
    rng = np.random.default_rng(seed)

    selected_ids = set()
    per_class_summary = []

    for cls in sorted(class_to_ids.keys()):
        cls_ids = list(class_to_ids[cls])
        n_take = min(images_per_class, len(cls_ids))

        if n_take > 0:
            chosen = rng.choice(cls_ids, size=n_take, replace=False).tolist()
        else:
            chosen = []

        selected_ids.update(chosen)

        per_class_summary.append({
            "class_name": cls,
            "available_images": len(cls_ids),
            "selected_images": len(chosen)
        })

    background_ids = sorted(set(background_ids or []))
    n_take_bg = min(images_per_class, len(background_ids))
    if n_take_bg > 0:
        chosen_bg = rng.choice(background_ids, size=n_take_bg, replace=False).tolist()
    else:
        chosen_bg = []
    selected_ids.update(chosen_bg)
    per_class_summary.append({
        "class_name": "No finding",
        "available_images": len(background_ids),
        "selected_images": len(chosen_bg),
    })

    return selected_ids, pd.DataFrame(per_class_summary)


rng_subset = np.random.default_rng(SEED)
background_pool = sorted(set(background_sample_ids))
if NO_FINDING_IMAGES is None:
    selected_background_ids = set(background_pool)
else:
    n_no_finding_take = min(int(NO_FINDING_IMAGES), len(background_pool))
    selected_background_ids = set(
        rng_subset.choice(background_pool, size=n_no_finding_take, replace=False).tolist()
    ) if n_no_finding_take > 0 else set()

if USE_SMALL_SUBSET:
    chosen_ids, subset_summary_df = build_small_subset_by_class_fast(
        class_to_ids,
        images_per_class=IMAGES_PER_CLASS,
        seed=SEED,
        background_ids=background_pool,
    )
    chosen_ids = set(map(str, chosen_ids)) & all_sample_ids
    print("Using SMALL subset mode")
    print("Total selected unique samples from merged CSV:", len(chosen_ids))
    display(subset_summary_df)
else:
    # Full mode: dùng đúng LABEL_CSV 1000nf giống train.py.
    # Nếu NO_FINDING_IMAGES=None thì giữ toàn bộ background đã có sẵn trong LABEL_CSV.
    chosen_ids = set(abnormal_sample_ids) | selected_background_ids
    print("Using FULL dataset mode from LABEL_CSV 1000nf")
    print("Abnormal samples selected:", len(abnormal_sample_ids))
    print("Background/No finding samples selected:", len(selected_background_ids), "of", len(background_pool))
    print("NO_FINDING_IMAGES:", NO_FINDING_IMAGES)
    print("Total selected unique samples from LABEL_CSV:", len(chosen_ids))

print("Matching requested DICOM files from both source folders...")

SOURCE_DIRS = {
    "train": TRAIN_DIR,
    "test": TEST_DIR,
}

sample_lookup_df = (
    df_csv_all[["sample_id", "source", "image_id"]]
    .drop_duplicates("sample_id")
    .reset_index(drop=True)
)


def find_dicom_records_for_sample_ids(sample_ids, sample_lookup_df, source_dirs):
    requested = set(map(str, sample_ids))
    lookup = sample_lookup_df[sample_lookup_df["sample_id"].isin(requested)].copy()

    records = []
    path_map = {}
    missing_sample_ids = []

    for source, group in lookup.groupby("source", sort=False):
        source = str(source).strip().lower()
        if source not in source_dirs:
            print(f"Unknown source={source}; mark samples as missing")
            missing_sample_ids.extend(group["sample_id"].astype(str).tolist())
            continue

        source_dir = Path(source_dirs[source])
        if not source_dir.exists():
            missing_sample_ids.extend(group["sample_id"].astype(str).tolist())
            continue

        image_ids = group["image_id"].astype(str).tolist()
        _, image_path_map, missing_image_ids = find_dicom_files_for_image_ids(source_dir, image_ids)
        missing_image_ids = set(missing_image_ids)

        for row in group.itertuples(index=False):
            sample_id = str(row.sample_id)
            image_id = str(row.image_id)
            if image_id in image_path_map:
                record = {
                    "sample_id": sample_id,
                    "source": str(row.source),
                    "image_id": image_id,
                    "path": image_path_map[image_id],
                }
                records.append(record)
                path_map[sample_id] = image_path_map[image_id]
            elif image_id in missing_image_ids:
                missing_sample_ids.append(sample_id)

    records = sorted(records, key=lambda x: x["sample_id"])
    missing_sample_ids = sorted(set(missing_sample_ids) | (requested - set(path_map.keys())))
    return records, path_map, missing_sample_ids


dicom_records_selected, dicom_path_map, missing_dicom_ids = find_dicom_records_for_sample_ids(
    chosen_ids,
    sample_lookup_df=sample_lookup_df,
    source_dirs=SOURCE_DIRS,
)
dicom_files_selected = [record["path"] for record in dicom_records_selected]
used_sample_ids = sorted(dicom_path_map.keys())
used_image_ids = used_sample_ids  # alias cũ, giờ là sample_id

print("Requested merged sample_ids:", len(chosen_ids))
print("Matched DICOM records:", len(dicom_records_selected))
print("Missing requested DICOM samples:", len(missing_dicom_ids))
if missing_dicom_ids:
    print("First missing sample_ids:", missing_dicom_ids[:10])

source_summary = pd.DataFrame(dicom_records_selected)["source"].value_counts().rename_axis("source").reset_index(name="matched_images")
display(source_summary)

In [ ]:
used_sample_ids = sorted(dicom_path_map.keys())
used_image_ids = used_sample_ids  # alias cũ, giờ là sample_id

df = df_full[df_full["sample_id"].astype(str).str.strip().isin(used_sample_ids)].copy()
used_negative_only_ids = sorted(set(used_sample_ids) - set(df["sample_id"].astype(str).unique()))

print("Requested merged sample_ids:", len(chosen_ids))
print("Used DICOM records:", len(dicom_records_selected))
print("Selected annotation rows:", len(df))
print("Used annotated sample_id:", df["sample_id"].nunique())
print("Used negative-only sample_id:", len(used_negative_only_ids))
print("Missing requested sample_ids:", len(missing_dicom_ids))

# Stratified split train/val/test

In [ ]:
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

BACKGROUND_CLASS = "No finding"


def normalize_split_ratios(split_ratios):
    split_ratios = dict(split_ratios)
    total = sum(float(v) for v in split_ratios.values())
    if total <= 0:
        raise ValueError("SPLIT_RATIOS phải có tổng > 0")
    return {k: float(v) / total for k, v in split_ratios.items()}


def compute_target_sizes(n_items, split_ratios):
    raw = {split: n_items * ratio for split, ratio in split_ratios.items()}
    sizes = {split: int(np.floor(value)) for split, value in raw.items()}
    remaining = n_items - sum(sizes.values())

    order = sorted(split_ratios.keys(), key=lambda split: (raw[split] - sizes[split], split), reverse=True)
    for split in order[:remaining]:
        sizes[split] += 1

    nonzero_splits = [split for split, ratio in split_ratios.items() if ratio > 0]
    if n_items >= len(nonzero_splits):
        for split in nonzero_splits:
            if sizes[split] == 0:
                donor = max(sizes, key=sizes.get)
                sizes[donor] -= 1
                sizes[split] += 1

    return sizes


def build_image_class_sets(df, image_ids, image_col="sample_id", class_col="class_name"):
    image_ids = sorted(set(map(str, image_ids)))
    grouped = (
        df.groupby(image_col)[class_col]
        .apply(lambda x: sorted(set(map(str, x))))
        .to_dict()
    )
    return {
        image_id: grouped.get(image_id, [BACKGROUND_CLASS])
        for image_id in image_ids
    }


def build_multilabel_stratified_split(df, image_ids, split_ratios=None, seed=42):
    split_ratios = normalize_split_ratios(split_ratios or {"train": 0.7, "val": 0.15, "test": 0.15})
    rng = np.random.default_rng(seed)

    image_ids = sorted(set(map(str, image_ids)))
    image_class_sets = build_image_class_sets(df, image_ids)
    split_names = list(split_ratios.keys())

    target_sizes = compute_target_sizes(len(image_ids), split_ratios)
    class_total = Counter(cls for classes in image_class_sets.values() for cls in classes)
    target_class_counts = {
        split: {cls: class_total[cls] * split_ratios[split] for cls in class_total}
        for split in split_names
    }

    current_sizes = {split: 0 for split in split_names}
    current_class_counts = {split: Counter() for split in split_names}
    split_sets = {split: set() for split in split_names}

    # Ưu tiên gán ảnh chứa class hiếm/multilabel trước để giữ phân bố class tốt hơn.
    ordered_ids = sorted(
        image_ids,
        key=lambda image_id: (
            -sum(1.0 / max(class_total[cls], 1) for cls in image_class_sets[image_id]),
            -len(image_class_sets[image_id]),
            rng.random(),
        ),
    )

    for image_id in ordered_ids:
        classes = image_class_sets[image_id]
        candidates = [split for split in split_names if current_sizes[split] < target_sizes[split]]
        if not candidates:
            candidates = split_names

        best_split = None
        best_score = None

        for split in candidates:
            size_deficit = (target_sizes[split] - current_sizes[split]) / max(target_sizes[split], 1)
            class_deficit = 0.0
            over_penalty = 0.0

            for cls in classes:
                target = max(target_class_counts[split][cls], 1e-9)
                before = current_class_counts[split][cls]
                after = before + 1
                class_deficit += max(target - before, 0.0) / target
                over_penalty += max(after - target, 0.0) / target

            score = size_deficit + class_deficit - over_penalty
            tie_breaker = rng.random() * 1e-6
            score += tie_breaker

            if best_score is None or score > best_score:
                best_score = score
                best_split = split

        split_sets[best_split].add(image_id)
        current_sizes[best_split] += 1
        for cls in classes:
            current_class_counts[best_split][cls] += 1

    return split_sets, image_class_sets


def summarize_split_distribution(split_sets, image_class_sets):
    class_names = sorted({cls for classes in image_class_sets.values() for cls in classes})
    rows = []
    for cls in class_names:
        total = sum(1 for classes in image_class_sets.values() if cls in classes)
        row = {"class_name": cls, "total_images": total}
        for split, ids in split_sets.items():
            count = sum(1 for image_id in ids if cls in image_class_sets[image_id])
            row[f"{split}_images"] = count
            row[f"{split}_pct_of_class"] = round(100.0 * count / total, 2) if total else 0.0
        rows.append(row)
    return pd.DataFrame(rows).sort_values("class_name").reset_index(drop=True)

In [ ]:
image_ids_selected = used_sample_ids.copy()

split_sets, image_class_sets = build_multilabel_stratified_split(
    df=df,
    image_ids=image_ids_selected,
    split_ratios=SPLIT_RATIOS,
    seed=SEED,
)

train_ids = split_sets["train"]
val_ids = split_sets["val"]
test_ids = split_sets["test"]
split_distribution_df = summarize_split_distribution(split_sets, image_class_sets)

print("Used images for split:", len(image_ids_selected))
print("Split ratios:", normalize_split_ratios(SPLIT_RATIOS))
print("Train images:", len(train_ids))
print("Val images:", len(val_ids))
print("Test images:", len(test_ids))
print("Overlap train/val:", len(train_ids & val_ids))
print("Overlap train/test:", len(train_ids & test_ids))
print("Overlap val/test:", len(val_ids & test_ids))

display(split_distribution_df)

# Lưu metadata để lần sau dùng lại

In [ ]:
import json

meta = {
    "label_csv": str(LABEL_CSV),
    "use_small_subset": USE_SMALL_SUBSET,
    "images_per_class": IMAGES_PER_CLASS,
    "no_finding_images": NO_FINDING_IMAGES,
    "selected_background_ids": sorted(list(selected_background_ids)),
    "seed": SEED,
    "split_ratios": normalize_split_ratios(SPLIT_RATIOS),
    "img_size": IMG_SIZE,
    "save_as_jpg": SAVE_AS_JPG,
    "jpg_quality": JPG_QUALITY,
    "preprocess_combos": [combo_to_name(combo) for combo in PREPROCESS_COMBOS],
    "enable_train_aug": ENABLE_TRAIN_AUG,
    "num_classes": len(CLASS_NAMES),
    "class_names": CLASS_NAMES,
    "wbf_iou_threshold": WBF_IOU_THRESHOLD,
    "wbf_skip_box_threshold": WBF_SKIP_BOX_THRESHOLD,
    "csv_bbox_format": CSV_BBOX_FORMAT,
    "chosen_sample_ids": sorted(list(chosen_ids)),
    "used_sample_ids": sorted(list(used_sample_ids)),
    "missing_dicom_sample_ids": sorted(list(missing_dicom_ids)),
    "train_ids": sorted(list(train_ids)),
    "val_ids": sorted(list(val_ids)),
    "test_ids": sorted(list(test_ids)),
}

OUT_ROOT.mkdir(parents=True, exist_ok=True)
with open(OUT_ROOT / "meta.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

split_distribution_df.to_csv(OUT_ROOT / "split_distribution.csv", index=False)

print("Saved:", OUT_ROOT / "meta.json")
print("Saved:", OUT_ROOT / "split_distribution.csv")

# Build annotation map nhanh

In [ ]:
ann_by_sample = {
    str(sample_id): group.copy()
    for sample_id, group in df.groupby("sample_id", sort=False)
}

split_map = {}
for x in train_ids:
    split_map[str(x)] = "train"
for x in val_ids:
    split_map[str(x)] = "val"
for x in test_ids:
    split_map[str(x)] = "test"

print("Images with annotations:", len(ann_by_sample))
print("Images without annotations:", len(set(split_map) - set(ann_by_sample)))
print("Images in split map:", len(split_map))
print("Split map counts:", pd.Series(split_map).value_counts().to_dict())

# Export ảnh + label nhanh, có track time

In [ ]:
import albumentations as A
import shutil
from collections import Counter

DICOM_TRAIN_AUG = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.UnsharpMask(
            blur_limit=(3, 5),
            sigma_limit=(0.1, 1.5),
            alpha=(0.1, 0.3),
            threshold=10,
            p=0.20,
        ),
        A.GaussianBlur(
            blur_limit=(3, 5),
            sigma_limit=(0.1, 1.5),
            p=0.20,
        ),
        A.Equalize(
            mode="cv",
            by_channels=False,
            p=0.20,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.10,
            contrast_limit=0.0,
            p=0.20,
        ),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["class_labels"],
        min_visibility=0.0,
    ),
)


def prepare_combo_output_dirs(combo_name):
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    combo_root = OUT_ROOT / combo_name
    if REBUILD_OUTPUT and combo_root.exists():
        shutil.rmtree(combo_root)

    paths = {
        "root": combo_root,
        "images": {},
        "labels": {},
    }
    for split in ["train", "val", "test"]:
        paths["images"][split] = combo_root / "images" / split
        paths["labels"][split] = combo_root / "labels" / split
        paths["images"][split].mkdir(parents=True, exist_ok=True)
        paths["labels"][split].mkdir(parents=True, exist_ok=True)
    return paths


def cleanup_combo_dataset(combo_paths):
    for key in ["images", "labels"]:
        root = combo_paths["root"] / key
        if root.exists():
            shutil.rmtree(root)
    print("Cleaned combo image/label dirs:", combo_paths["root"])


def sanitize_pascal_voc_boxes(boxes, width, height):
    sanitized = []
    for (x1, y1, x2, y2, class_id) in boxes:
        x1 = float(np.clip(x1, 0, width))
        y1 = float(np.clip(y1, 0, height))
        x2 = float(np.clip(x2, 0, width))
        y2 = float(np.clip(y2, 0, height))

        if x2 <= x1 or y2 <= y1:
            continue

        sanitized.append((x1, y1, x2, y2, class_id))

    return sanitized


def apply_dicom_augmentation(img, boxes, split):
    height, width = img.shape[:2]
    boxes = sanitize_pascal_voc_boxes(boxes, width=width, height=height)

    if split != "train" or len(boxes) == 0 or not ENABLE_TRAIN_AUG:
        return img, boxes

    bboxes = [(x1, y1, x2, y2) for (x1, y1, x2, y2, _) in boxes]
    class_labels = [class_id for (_, _, _, _, class_id) in boxes]

    transformed = DICOM_TRAIN_AUG(
        image=img,
        bboxes=bboxes,
        class_labels=class_labels,
    )

    aug_boxes = [
        (x1, y1, x2, y2, class_id)
        for (x1, y1, x2, y2), class_id in zip(
            transformed["bboxes"],
            transformed["class_labels"],
        )
    ]
    aug_boxes = sanitize_pascal_voc_boxes(aug_boxes, width=width, height=height)
    return transformed["image"], aug_boxes


def sample_source_size(rows):
    if rows is None or len(rows) == 0:
        return None
    widths = pd.to_numeric(rows.get("source_width", pd.Series(dtype=float)), errors="coerce").dropna()
    heights = pd.to_numeric(rows.get("source_height", pd.Series(dtype=float)), errors="coerce").dropna()
    if len(widths) > 0 and len(heights) > 0 and float(widths.iloc[0]) > 0 and float(heights.iloc[0]) > 0:
        return float(widths.iloc[0]), float(heights.iloc[0])
    return None


def detect_bbox_format(rows, source_size, actual_size):
    if CSV_BBOX_FORMAT != "auto":
        return CSV_BBOX_FORMAT
    if rows is None or len(rows) == 0:
        return "xyxy_pixel"

    coords = rows[["x_min", "y_min", "x_max", "y_max"]].to_numpy(dtype=float)
    finite = coords[np.isfinite(coords)]
    if finite.size and np.nanmax(finite) <= 1.05 and np.nanmin(finite) >= -0.05:
        return "xyxy_norm"

    if source_size is not None:
        source_w, source_h = source_size
        max_x = np.nanmax(coords[:, [0, 2]]) if coords.size else 0.0
        max_y = np.nanmax(coords[:, [1, 3]]) if coords.size else 0.0
        if max_x > source_w * 1.25 or max_y > source_h * 1.25:
            raise ValueError(
                "CSV bbox lớn hơn nhiều so với source image size tìm được. "
                f"bbox_max=({max_x:.1f}, {max_y:.1f}), source_size=({source_w:.1f}, {source_h:.1f})."
            )
        return "xyxy_pixel"

    actual_w, actual_h = actual_size
    max_x = np.nanmax(coords[:, [0, 2]]) if coords.size else 0.0
    max_y = np.nanmax(coords[:, [1, 3]]) if coords.size else 0.0
    if max_x > actual_w * 1.25 or max_y > actual_h * 1.25:
        raise ValueError(
            "CSV bbox có vẻ là pixel theo ảnh gốc nhưng không khớp DICOM size. "
            "Thêm image_width/image_height vào CSV hoặc kiểm tra bbox source."
        )
    return "xyxy_pixel"


def row_to_yolo_line(row, class2id, source_size, actual_size, bbox_format):
    cls_idx = class2id[str(row["class_name"])]
    x1 = float(row["x_min"])
    y1 = float(row["y_min"])
    x2 = float(row["x_max"])
    y2 = float(row["y_max"])

    if bbox_format in {"xyxy_norm", "normalized_xyxy"}:
        x1 = float(np.clip(x1, 0.0, 1.0))
        y1 = float(np.clip(y1, 0.0, 1.0))
        x2 = float(np.clip(x2, 0.0, 1.0))
        y2 = float(np.clip(y2, 0.0, 1.0))
        xc = (x1 + x2) / 2.0
        yc = (y1 + y2) / 2.0
        bw = x2 - x1
        bh = y2 - y1
    elif bbox_format in {"xyxy_pixel", "pixel_xyxy"}:
        denom_w, denom_h = source_size if source_size is not None else actual_size
        x1 = float(np.clip(x1, 0.0, denom_w))
        y1 = float(np.clip(y1, 0.0, denom_h))
        x2 = float(np.clip(x2, 0.0, denom_w))
        y2 = float(np.clip(y2, 0.0, denom_h))
        xc = ((x1 + x2) / 2.0) / denom_w
        yc = ((y1 + y2) / 2.0) / denom_h
        bw = (x2 - x1) / denom_w
        bh = (y2 - y1) / denom_h
    else:
        raise ValueError(f"CSV_BBOX_FORMAT không hỗ trợ: {CSV_BBOX_FORMAT}")

    xc = float(np.clip(xc, 0.0, 1.0))
    yc = float(np.clip(yc, 0.0, 1.0))
    bw = float(np.clip(bw, 0.0, 1.0))
    bh = float(np.clip(bh, 0.0, 1.0))
    if bw <= 0.0 or bh <= 0.0:
        return None
    return f"{cls_idx} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"


def build_yolo_lines_from_rows(rows, class2id, source_size, actual_size):
    lines = []
    bbox_format = detect_bbox_format(rows, source_size, actual_size)
    if rows is None or len(rows) == 0:
        return lines, bbox_format
    for _, row in rows.iterrows():
        line = row_to_yolo_line(row, class2id, source_size, actual_size, bbox_format)
        if line is not None:
            lines.append(line)
    return lines, bbox_format


def rows_to_pascal_voc_boxes(rows, class2id, source_size, actual_size):
    if rows is None or len(rows) == 0:
        return []

    bbox_format = detect_bbox_format(rows, source_size, actual_size)
    actual_w, actual_h = actual_size
    boxes = []

    for _, row in rows.iterrows():
        class_id = class2id[str(row["class_name"])]
        x1 = float(row["x_min"])
        y1 = float(row["y_min"])
        x2 = float(row["x_max"])
        y2 = float(row["y_max"])

        if bbox_format in {"xyxy_norm", "normalized_xyxy"}:
            x1 *= actual_w
            x2 *= actual_w
            y1 *= actual_h
            y2 *= actual_h
        else:
            denom_w, denom_h = source_size if source_size is not None else actual_size
            x1 = float(np.clip(x1, 0.0, denom_w)) * actual_w / denom_w
            x2 = float(np.clip(x2, 0.0, denom_w)) * actual_w / denom_w
            y1 = float(np.clip(y1, 0.0, denom_h)) * actual_h / denom_h
            y2 = float(np.clip(y2, 0.0, denom_h)) * actual_h / denom_h

        boxes.append((x1, y1, x2, y2, class_id))

    return sanitize_pascal_voc_boxes(boxes, width=actual_w, height=actual_h)


def build_yolo_lines_from_pascal_boxes(boxes, actual_size):
    actual_w, actual_h = actual_size
    lines = []
    for x1, y1, x2, y2, class_id in boxes:
        x1 = float(np.clip(x1, 0.0, actual_w))
        x2 = float(np.clip(x2, 0.0, actual_w))
        y1 = float(np.clip(y1, 0.0, actual_h))
        y2 = float(np.clip(y2, 0.0, actual_h))
        bw = (x2 - x1) / actual_w
        bh = (y2 - y1) / actual_h
        if bw <= 0.0 or bh <= 0.0:
            continue
        xc = ((x1 + x2) / 2.0) / actual_w
        yc = ((y1 + y2) / 2.0) / actual_h
        lines.append(f"{int(class_id)} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    return lines


def write_label_file(label_path: Path, yolo_lines):
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(yolo_lines) + ("\n" if yolo_lines else ""))


def save_combo_image_and_label(
    img,
    rows,
    sample_id,
    split,
    combo_paths,
    actual_size,
    source_size=None,
    boxes_override=None,
):
    img_resized, _, _ = resize_image_keep_shape(img, size=IMG_SIZE)
    ext = ".jpg" if SAVE_AS_JPG else ".png"

    img_out_path = combo_paths["images"][split] / f"{sample_id}{ext}"
    lbl_out_path = combo_paths["labels"][split] / f"{sample_id}.txt"

    # Treat stacked methods as RGB-like channels on disk.
    # OpenCV writes 3-channel arrays as BGR, so reverse before imwrite.
    img_to_write = img_resized
    if img_to_write.ndim == 3 and img_to_write.shape[2] == 3:
        img_to_write = cv2.cvtColor(img_to_write, cv2.COLOR_RGB2BGR)

    if SAVE_AS_JPG:
        cv2.imwrite(str(img_out_path), img_to_write, [cv2.IMWRITE_JPEG_QUALITY, JPG_QUALITY])
    else:
        cv2.imwrite(str(img_out_path), img_to_write, [cv2.IMWRITE_PNG_COMPRESSION, 0])

    if boxes_override is not None:
        yolo_lines = build_yolo_lines_from_pascal_boxes(boxes_override, actual_size=actual_size)
        bbox_format = "aug_pascal_voc"
    else:
        yolo_lines, bbox_format = build_yolo_lines_from_rows(
            rows=rows,
            class2id=CLASS2ID,
            source_size=source_size,
            actual_size=actual_size,
        )

    write_label_file(lbl_out_path, yolo_lines)
    return len(yolo_lines), bbox_format


def process_one_dicom_for_combo(record, combo, combo_paths):
    t0 = time.perf_counter()

    sample_id = str(record["sample_id"])
    dicom_path = Path(record["path"])
    split = split_map.get(sample_id)
    if split is None:
        return {
            "image_id": sample_id,
            "ok": False,
            "reason": "not_in_split",
            "elapsed": 0.0,
            "num_saved": 0,
            "label_count": 0,
            "bbox_format_counts": {},
        }

    rows = ann_by_sample.get(sample_id, pd.DataFrame(columns=df.columns))

    try:
        variants = read_dicom_preprocess_variants(dicom_path)
        first_variant = next(iter(variants.values()))
        orig_h, orig_w = first_variant.shape[:2]
        actual_size = (orig_w, orig_h)
        source_size = sample_source_size(rows)

        base_boxes = rows_to_pascal_voc_boxes(
            rows=rows,
            class2id=CLASS2ID,
            source_size=source_size,
            actual_size=actual_size,
        )

        img_combo = build_preprocess_combo_image(variants, combo)
        num_saved = 0
        label_count = 0
        bbox_format_counts = Counter()

        n_labels, bbox_format = save_combo_image_and_label(
            img=img_combo,
            rows=rows,
            sample_id=sample_id,
            split=split,
            combo_paths=combo_paths,
            actual_size=actual_size,
            source_size=source_size,
        )
        num_saved += 1
        label_count += n_labels
        bbox_format_counts[bbox_format] += 1

        if split == "train" and len(base_boxes) > 0 and ENABLE_TRAIN_AUG:
            img_aug, boxes_aug = apply_dicom_augmentation(
                img_combo.copy(),
                base_boxes,
                split,
            )
            n_aug_labels, aug_bbox_format = save_combo_image_and_label(
                img=img_aug,
                rows=rows,
                sample_id=f"{sample_id}_aug",
                split=split,
                combo_paths=combo_paths,
                actual_size=actual_size,
                source_size=source_size,
                boxes_override=boxes_aug,
            )
            num_saved += 1
            label_count += n_aug_labels
            bbox_format_counts[aug_bbox_format] += 1

        elapsed = time.perf_counter() - t0
        return {
            "image_id": sample_id,
            "ok": True,
            "reason": "done",
            "elapsed": elapsed,
            "num_saved": num_saved,
            "label_count": label_count,
            "bbox_format_counts": dict(bbox_format_counts),
        }

    except Exception as e:
        elapsed = time.perf_counter() - t0
        return {
            "image_id": sample_id,
            "ok": False,
            "reason": str(e),
            "elapsed": elapsed,
            "num_saved": 0,
            "label_count": 0,
            "bbox_format_counts": {},
        }


def write_combo_yaml(combo_paths):
    data_yaml = {
        "path": str(combo_paths["root"]),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }

    yaml_path = combo_paths["root"] / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f, sort_keys=False)
    return yaml_path


def export_combo_dataset(combo, combo_paths):
    combo_name = combo_to_name(combo)
    max_workers = min(8, os.cpu_count() or 4)
    print("max_workers =", max_workers)
    print("Export combo:", combo_name)

    total_files = len(dicom_records_selected)
    start_all = time.perf_counter()

    done = 0
    ok_count = 0
    times = []
    label_count = 0
    bbox_format_counts = Counter()
    failed_results = []
    log_every = 20

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_one_dicom_for_combo, record, combo, combo_paths) for record in dicom_records_selected]

        for future in as_completed(futures):
            result = future.result()
            done += 1

            if result["ok"]:
                ok_count += 1
                times.append(result["elapsed"])
                label_count += int(result.get("label_count", 0))
                bbox_format_counts.update(result.get("bbox_format_counts", {}))
            else:
                failed_results.append(result)

            if done % log_every == 0 or done == total_files:
                elapsed_all = time.perf_counter() - start_all
                avg_per_file = elapsed_all / done
                speed = done / elapsed_all if elapsed_all > 0 else 0.0
                remaining = total_files - done
                eta_sec = remaining * avg_per_file
                avg_worker = np.mean(times) if len(times) > 0 else 0.0
                p50_worker = np.median(times) if len(times) > 0 else 0.0

                print(
                    f"[{done}/{total_files}] "
                    f"ok={ok_count} | "
                    f"wall={elapsed_all:.1f}s | "
                    f"speed={speed:.2f} dicom/s | "
                    f"avg_wall/dicom={avg_per_file:.3f}s | "
                    f"avg_worker={avg_worker:.3f}s | "
                    f"p50_worker={p50_worker:.3f}s | "
                    f"ETA={eta_sec:.1f}s"
                )

    total_elapsed = time.perf_counter() - start_all
    img_ext = "*.jpg" if SAVE_AS_JPG else "*.png"
    split_image_counts = {
        split: len(list(combo_paths["images"][split].glob(img_ext)))
        for split in ["train", "val", "test"]
    }
    split_label_counts = {
        split: len(list(combo_paths["labels"][split].glob("*.txt")))
        for split in ["train", "val", "test"]
    }

    stats = {
        "combo": combo_name,
        "export_processed_dicom": done,
        "export_success_dicom": ok_count,
        "export_failed_dicom": len(failed_results),
        "export_seconds": total_elapsed,
        "export_label_count": label_count,
        "bbox_format_counts": dict(bbox_format_counts),
        "train_images_exported": split_image_counts["train"],
        "val_images_exported": split_image_counts["val"],
        "test_images_exported": split_image_counts["test"],
        "train_labels_exported": split_label_counts["train"],
        "val_labels_exported": split_label_counts["val"],
        "test_labels_exported": split_label_counts["test"],
    }
    if failed_results:
        print("First failed results:")
        display(pd.DataFrame(failed_results).head(10))
    return stats

# Chạy export song song + ETA

In [ ]:
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

combo_pipeline_rows = []
trained_run_info = {}
summary_path = OUT_ROOT / "sequential_combo_results.csv"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

for combo in PREPROCESS_COMBOS:
    combo_name = combo_to_name(combo)
    combo_paths = prepare_combo_output_dirs(combo_name)
    yaml_path = None
    row = {
        "combo": combo_name,
        "status": "started",
    }

    print("\n" + "=" * 100)
    print("Combo:", combo_name)
    print("Dataset root:", combo_paths["root"])

    try:
        export_stats = export_combo_dataset(combo, combo_paths)
        row.update(export_stats)

        yaml_path = write_combo_yaml(combo_paths)
        row["yaml_path"] = str(yaml_path)
        print("Saved YAML:", yaml_path)

        run_name = f"{Path(MODEL_NAME).stem}_{combo_name}"
        row["run_name"] = run_name
        print("Training run:", run_name)

        model = YOLO(MODEL_NAME)
        model.train(
            data=str(yaml_path),
            epochs=TRAIN_EPOCHS,
            imgsz=IMG_SIZE,
            batch=TRAIN_BATCH,
            device=DEVICE,
            project=str(OUT_ROOT / "runs"),
            name=run_name,
            exist_ok=True,
        )

        best_weight_path = OUT_ROOT / "runs" / run_name / "weights" / "best.pt"
        row["best_weight_path"] = str(best_weight_path)
        trained_run_info[combo_name] = {
            "run_name": run_name,
            "yaml_path": str(yaml_path),
            "best_weight_path": str(best_weight_path),
        }

        if not best_weight_path.exists():
            raise FileNotFoundError(f"Không thấy best weight sau train: {best_weight_path}")

        print("Testing combo:", combo_name)
        best_model = YOLO(str(best_weight_path))
        metrics = best_model.val(
            data=str(yaml_path),
            split="test",
            imgsz=IMG_SIZE,
            batch=EVAL_BATCH,
            device=DEVICE,
            verbose=True,
        )

        row.update({
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "map50": float(metrics.box.map50),
            "map50_95": float(metrics.box.map),
            "status": "done",
        })

    except Exception as e:
        row["status"] = "failed"
        row["error"] = str(e)
        print("Combo failed:", combo_name)
        print(e)

    finally:
        combo_pipeline_rows.append(row)
        combo_pipeline_results_df = pd.DataFrame(combo_pipeline_rows)
        combo_pipeline_results_df.to_csv(summary_path, index=False)
        print("Saved running summary:", summary_path)
        display(combo_pipeline_results_df.tail(1))

        if CLEANUP_AFTER_COMBO:
            cleanup_combo_dataset(combo_paths)

    if row.get("status") != "done":
        raise RuntimeError(f"Dừng pipeline vì combo failed: {combo_name}")

print("\nAll combos finished")
print("Saved summary:", summary_path)
display(combo_pipeline_results_df)

In [ ]:
print("Sequential combo pipeline summary")
print("Summary CSV:", OUT_ROOT / "sequential_combo_results.csv")
display(combo_pipeline_results_df)

# Tạo data.yaml

In [ ]:
# data.yaml được tạo tạm cho từng combo trong sequential pipeline cell.
# Sau khi train/eval xong, images/labels của combo được xóa nếu CLEANUP_AFTER_COMBO=True.
print("YAML creation is handled inside the sequential combo pipeline.")
print("CLEANUP_AFTER_COMBO:", CLEANUP_AFTER_COMBO)

In [ ]:
# Preview chỉ dùng được nếu CLEANUP_AFTER_COMBO=False hoặc chạy riêng một combo và chưa cleanup.
print("Preview skipped because sequential mode cleans combo images/labels to save Kaggle disk.")
print("Set CLEANUP_AFTER_COMBO=False if you need to preview exported images before cleanup.")

In [ ]:
# Training đã được xử lý trong sequential combo pipeline cell.
print("Training already completed in the sequential combo pipeline cell.")
display(combo_pipeline_results_df)

In [ ]:
# Eval metrics đã được xử lý trong sequential combo pipeline cell.
metrics_path = OUT_ROOT / "sequential_combo_results.csv"
print("Saved metrics:", metrics_path)
display(pd.read_csv(metrics_path))